# 01 — Cohort and temporal split

## Purpose

This notebook assigns each admission to a chronological `train`, `validation`, or `test` period.

The split is based only on `ADMISSION_DATE` and the fixed dates in `analysis.yaml`.

- `train`: admission date < `train_end`
- `validation`: `train_end` ≤ admission date < `validation_end`
- `test`: admission date ≥ `validation_end`

Rows with missing or unparseable admission dates are retained in the table but receive a missing split label and are reported. The temporal cut-offs are study-design decisions and must not be chosen using test-set performance.


In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from pathlib import Path
import yaml
import pandas as pd
from icare_risk.data.audit import load_admission_file
from icare_risk.data.validation import validate_admission_table, validate_binary_target
from icare_risk.data.temporal_split import add_split_label, split_summary

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as file:
    config = yaml.safe_load(file)

columns = config["columns"]
input_cfg = config["input"]
analysis_cfg = config["analysis"]
target_mapping = config.get(
    "target_mapping",
    analysis_cfg.get("target_mapping", {}),
)

train_end = pd.Timestamp(analysis_cfg["train_end"])
validation_end = pd.Timestamp(analysis_cfg["validation_end"])

if train_end >= validation_end:
    raise ValueError(
        f"Invalid temporal boundaries: train_end={train_end.date()} "
        f"must be earlier than validation_end={validation_end.date()}."
    )

print("Configuration")
print(f"  Config file: {CONFIG_PATH}")
print(f"  Input file: {input_cfg['path']}")
print(f"  Admission date: {columns['admission_date']}")
print(f"  Target: {columns['target']}")
print(f"  Train end: {train_end.date()}")
print(f"  Validation end: {validation_end.date()}")
print(f"  Target mapping: {target_mapping}")
print("  Test admissions will not be used for model fitting.")

Configuration
  Config file: ../src/icare_risk/config/icare/analysis.yaml
  Input file: ../notebooks/data/synthetic/10000p/03_derived_analysis_dataset.parquet
  Admission date: ADMISSION_DATE
  Target: esbl_label_test
  Train end: 2022-06-01
  Validation end: 2022-12-31
  Target mapping: {'positive_values': ['esbl', 'positive', 1, '1'], 'negative_values': ['non-esbl', 'non esbl', 'negative', 0, '0']}
  Test admissions will not be used for model fitting.


In [16]:
df = load_admission_file(
    input_cfg["path"],
    **input_cfg.get("read_csv_kwargs", {}),
)

required_columns = config.get("validation", {}).get("required_columns", [])
structure_validation = validate_admission_table(
    df=df,
    required_columns=required_columns,
    admission_id=columns["admission_id"],
    date_columns=[
        columns["admission_date"],
        columns["discharge_date"]
    ],
)
target_validation = validate_binary_target(
    df=df,
    target_column=columns["target"],
)

parsed_dates = pd.to_datetime(
    df[columns["admission_date"]],
    errors="coerce",
)

print("Input dataset audit")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {df.shape[1]:,}")
print(f"  Earliest admission: {parsed_dates.min()}")
print(f"  Latest admission: {parsed_dates.max()}")
print(
    "  Missing/unparseable admission dates: "
    f"{parsed_dates.isna().sum():,}"
)
print(
    "  Duplicate admission identifiers: "
    f"{structure_validation['duplicate_admission_rows']:,}"
)
print("  Target values:")
display(df[columns["target"]].value_counts(dropna=False).rename("n"))
print("  Structural validation:")
display(pd.DataFrame([structure_validation]))
print("  Target validation:")
display(pd.DataFrame([target_validation]))


Input dataset audit
  Rows: 10,000
  Columns: 186
  Earliest admission: 2020-01-01 00:00:00
  Latest admission: 2024-01-01 00:00:00
  Missing/unparseable admission dates: 0
  Duplicate admission identifiers: 14
  Target values:


esbl_label_test
0    8042
1    1958
Name: n, dtype: int64

  Structural validation:


,n_rows,n_columns,is_empty,admission_id_present,duplicate_admission_rows,date_parse_failures,warnings
0,10000,186,False,True,14,"{'ADMISSION_DATE': 0, 'DISCHARGE_DATE': 0}",[Duplicate values found in SPELL_IDENTIFIER]


  Target validation:


,target_column,n_missing,unique_values,n_unique_non_missing,is_binary
0,esbl_label_test,0,"[1, 0]",2,True


In [17]:
df = add_split_label(
    df=df,
    date_column=columns["admission_date"],
    train_end=train_end,
    validation_end=validation_end,
    output_column="split",
)

summary = split_summary(
    df=df,
    split_column="split",
    target_column=columns["target"],
    target_mapping=target_mapping,
)

print("Temporal split summary")
display(summary)

missing_split = int(df["split"].isna().sum())
print(f"Rows without a split: {missing_split:,}")

if missing_split:
    print(
        "These rows have missing or unparseable admission dates and will be "
        "excluded from downstream train/validation/test analyses unless handled explicitly."
    )

split_counts = df["split"].value_counts(dropna=False)
print("Rows by split:")
display(split_counts.rename("n_admissions").to_frame())

Temporal split summary


,split,n_admissions,n_non_missing_target,n_unknown_target,n_events,event_rate
0,train,6054,6054,0,1186,0.195904
1,test,2483,2483,0,493,0.198550
2,validation,1463,1463,0,279,0.190704


Rows without a split: 0
Rows by split:


,n_admissions
split,
train,6054
test,2483
validation,1463


In [18]:
# Save the labelled admission-level dataset for all downstream notebooks.
output_path = Path(config['outputs']['processed_dataset'])
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(output_path, index=False)
print(f'Saved {len(df):,} admissions to {output_path}')

Saved 10,000 admissions to data/processed/admissions_with_split.parquet
